# 💧 Smart Irrigation - Data Cleaning, Preprocessing & Feature Engineering Notebook
**Project:** Priority-Based Machine Learning Smart Irrigation Decision Support System
**Target Reservoir:** Nachchaduwa / Mahakandarawa Reservoir Data
**Output File:** `reservoir_data_with_lag_features.xlsx`

### Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print('Libraries imported successfully!')

### Step 2: Load Raw Reservoir Dataset
Upload your raw Excel or CSV dataset to Colab and load it.

In [ ]:
# If uploading file to Colab:
# from google.colab import files
# uploaded = files.upload()

# Load dataset
file_path = 'reservoir_data_with_lag_features.xlsx'  # or raw_reservoir_data.xlsx
df = pd.read_excel(file_path)

print('Initial Dataset Shape:', df.shape)
df.head()

### Step 3: Data Cleaning & Handling Missing Values
- Date parsing and chronological sorting
- Duplicates check and removal
- Missing value treatment (Interpolation and Forward Fill)

In [ ]:
# 1. Ensure Date column is in datetime format
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# 2. Drop duplicates
df = df.drop_duplicates(subset=['Date']).reset_index(drop=True)

# 3. Numeric columns cleaning and missing value interpolation
numeric_cols = [
    'Reservoir Water Level',
    'Reservoir Capacity',
    'Previous Water Release',
    'Rainfall (Nachchaduwa)'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        # Linearly interpolate missing values, then forward fill/bfill
        df[col] = df[col].interpolate(method='linear').ffill().bfill()

# 4. Add data quality flag
df['data_quality_flag'] = 'VALID'

print('Cleaned Missing Values!')
df.info()

### Step 4: Feature Engineering (Creating Lag & Cumulative Features)
Create temporal lag features and rolling cumulative rainfall sums:
1. `prev_day_level`: Water level of previous day ($t-1$)
2. `prev_day_release`: Water release of previous day ($t-1$)
3. `level_change`: Daily water level change ($Level_t - Level_{t-1}$)
4. `rainfall_3day_sum`: 3-day accumulated rolling rainfall
5. `rainfall_7day_sum`: 7-day accumulated rolling rainfall

In [ ]:
# 1. Lag Features (Previous Day Water Level & Previous Day Release)
df['prev_day_level'] = df['Reservoir Water Level'].shift(1)
df['prev_day_release'] = df['Previous Water Release'].shift(1)

# 2. Water Level Change Feature
df['level_change'] = df['Reservoir Water Level'] - df['prev_day_level']

# 3. Rolling Cumulative Rainfall Features (3-Day & 7-Day Sums)
df['rainfall_3day_sum'] = df['Rainfall (Nachchaduwa)'].rolling(window=3, min_periods=1).sum()
df['rainfall_7day_sum'] = df['Rainfall (Nachchaduwa)'].rolling(window=7, min_periods=1).sum()

# Fill initial NaN created by shift with backward fill
df['prev_day_level'] = df['prev_day_level'].bfill()
df['prev_day_release'] = df['prev_day_release'].bfill()
df['level_change'] = df['level_change'].fillna(0.0)

print('Feature Engineering Completed!')
df[['Date', 'Reservoir Water Level', 'prev_day_level', 'level_change', 'rainfall_3day_sum']].head(10)

### Step 5: Export Cleaned & Preprocessed Dataset to Excel
Save the final feature-engineered dataset to `reservoir_data_with_lag_features.xlsx`.

In [ ]:
# Final column arrangement
final_columns = [
    'Date',
    'Reservoir Water Level',
    'Reservoir Capacity',
    'Previous Water Release',
    'Rainfall (Nachchaduwa)',
    'data_quality_flag',
    'prev_day_level',
    'prev_day_release',
    'rainfall_3day_sum',
    'rainfall_7day_sum',
    'level_change'
]

final_df = df[final_columns]

# Save to Excel
output_filename = 'reservoir_data_with_lag_features.xlsx'
final_df.to_excel(output_filename, index=False)

print(f'Successfully exported final preprocessed dataset: {output_filename}')
print('Final Shape:', final_df.shape)

# Download file from Google Colab automatically
# from google.colab import files
# files.download(output_filename)